# Orbit Wars RL Reward Sistemi

Bu notebook, Orbit Wars PPO eğitiminde kullanılan reward sistemini uçtan uca açıklar.

Ana kaynak dosyalar:

- `rl/rewards.py`: `RewardConfig`, step-level shaping ve reward presetleri.
- `rl/rollout.py`: environment step, transition üretimi ve rewardların buffer'a yazılması.
- `rl/fleet_tracker.py`: fleet arrival/outcome takip ve delayed fleet reward.
- `rl/production_reward_tracker.py`: `production_simple` capture, neutral gift ve production-loss attribution.
- `rl/defense_tracker.py`: no-op/hold defense attribution.
- `rl/ppo.py`: PPO loss içinde rewardların advantage/return olarak kullanılması.


## 1. Reward Sisteminin Büyük Resmi

Training sırasında tek bir environment step'inde birden fazla reward kaynağı oluşabilir. Sistem özellikle **action-specific delayed credit assignment** üzerine kurulu. Yani bir capture, fleet arrival veya defense olayı gerçekleştiğinde reward mümkün olduğunca onu başlatan eski transition'a geri yazılır.

Genel akış:

```text
collect_episode
  ├─ env_obs alınır
  ├─ her oyuncu için _act_for_player çağrılır
  ├─ policy actionları env action formatına decode edilir
  ├─ env.step(env_actions) çalışır
  ├─ immediate / step-level rewards hesaplanır
  ├─ transitionlar RolloutBuffer'a eklenir
  ├─ launch metadata tracker'lara kaydedilir
  └─ önceki launch/hold olayları resolve olursa delayed reward buffer_idx'ye eklenir
```

Kritik tasarım kararı: capture gibi sonuçlar oyundan birkaç turn sonra gelebilir; bu reward o andaki tüm actionlara yayılmaz, **orijinal launch transition**'ına gider.


## 2. RewardConfig

`RewardConfig`, tüm reward ağırlıklarının merkezi konfigürasyonudur. Training CLI'daki reward flagleri `build_reward_config(...)` üzerinden bu dataclass'a dönüşür.

Önemli alanlar:

| Alan | Anlam | Tipik rol |
|---|---|---|
| `reward_system` | `current`, `aggressive`, `production_simple` | Preset seçimi |
| `max_turns` | Time decay için oyun uzunluğu | Genelde `500` |
| `fleet_airborne_weight` | Launch anında fleet'i havaya göndermenin maliyeti | Early reckless launch'ı cezalandırır |
| `fleet_kill_weight` | Enemy ship öldürme rewardu | Combat signal |
| `fleet_waste_weight` | Boşa giden fleet cezası | Failed attack cezası |
| `planet_loss_weight` | Defense deficit / planet loss penalty | Savunmasız bırakmayı cezalandırır |
| `delayed_reward_discount` | Delayed credit gecikme discountu | `reward *= discount ** delay` |
| `reward_scale` | Production-simple reward ölçeği | Genelde `0.001` |
| `draw_terminal_reward` | Draw terminal rewardu | `production_simple` için `-1.0` |
| `production_reward_beta` | Production değerinin superlinear exponent'i | Deneysel |
| `production_reward_reference` | Beta uygulandığında sabit kalan referans üretim | Genelde `3.0` |


## 3. Reward Presetleri

### `current`

Backward-compatible varsayılan sistemdir. Step-level production swing, ship differential, fleet capture/kill/waste ve defense hold gibi klasik shaping bileşenlerini kullanır.

### `aggressive`

`current` sisteminin daha agresif ağırlıklandırılmış hali. Denial ve enemy capture ağırlıkları daha yüksektir.

### `production_simple`

Aktif deneylerin çoğunda kullanılan sistemdir. Step-level capture-value shaping kapatılır ve ana öğrenme sinyali launch-specific delayed production payback'e taşınır.

Öne çıkan varsayılanları:

```text
capture_value_weight = 0.0
denial_value_weight = 0.0
ship_diff_weight = 0.0
ship_loss_weight = 0.0
fleet_airborne_weight = 1.0
fleet_kill_weight = 1.0
fleet_waste_weight = 1.0
planet_loss_weight = 1.0
reward_scale = 0.001
draw_terminal_reward = -1.0
```

Bu preset'te rewardlar büyük ölçüde `production_reward_tracker.py` ve `fleet_tracker.py` üzerinden action'a geri bağlanır.


## 4. Step-Level Reward

`compute_step_reward(prev_env_obs, curr_env_obs, player, ...)` her env step sonrası çağrılır. Bu reward, o step'te oyuncu için oluşturulan transitionlara eşit bölünür.

Formülün ana hali:

```text
time_factor = max(time_decay_floor, (max_turns - turn) / max_turns)

gain   = time_factor * Δ(my_planet_value_total)
denial = time_factor * -Δ(opponent_planet_value_total)

shaped = capture_value_weight * gain
       + denial_value_weight * denial
       + planet_diff_weight * d_planets
       + ship_diff_weight * d_ship_diff
       - ship_loss_weight * ship_loss
       - invalid_action_penalty * invalid_actions
       + step_cost
```

`production_simple` içinde bu step-level production/ship shaping ağırlıkları `0.0` olduğu için ana signal delayed launch rewarddan gelir.


## 5. Launch-Time Airborne Cost

Bir fleet launch edildiği anda, daha outcome beklenmeden launch transition'a bir airborne cost yazılır. Bu, shiplerin bir süre kullanılamaması ve uzun/uzak saldırıların fırsat maliyetini temsil eder.

`rl/rollout.py` içinde:

```text
airborne_cost = -fleet_airborne_weight * ships_sent * eta
airborne_cost *= risk_multiplier
airborne_cost *= reward_scale
```

`risk_multiplier`, source planet enemy incoming threat altındaysa maliyeti artırabilir. Böylece agent, savunması düşmek üzere olan planetten büyük fleet çıkarmayı daha riskli görür.


## 6. Production Capture Reward

`production_simple` sisteminin ana rewardu capture payback'tir. `ProductionRewardTracker`, launchları target planet bazında takip eder ve planet owner oyuncuya döndüğünde eligible launchlara reward yazar.

Temel formül:

```text
remaining = max_turns - capture_turn
production_value = production_reward_value(planet_production)

base = production_value * remaining
if captured_from_enemy:
    base *= opponent_count_multiplier(num_opponents)

base -= total_ship_investment

reward = base * (ships_sent / total_ships_sent) * reward_scale
```

Buradaki `total_ship_investment`, target'ın başlangıç/observed ship maliyetidir. Aynı hedefe birden fazla fleet capture'a katkı sağlarsa reward `ships_sent / total_ships_sent` oranıyla bölünür.

Enemy-owned planet capture, neutral capture'dan daha değerli olur çünkü rakipten almak production swing etkisi yaratır ve `opponent_count_multiplier` uygular.


In [ ]:
def production_reward_value(production, beta=0.0, reference=3.0):
    production = max(0.0, float(production))
    beta = max(0.0, float(beta))
    if production <= 0.0 or beta <= 0.0:
        return production
    return production * ((production / reference) ** beta)

for beta in [0.0, 0.2, 0.5]:
    values = [round(production_reward_value(p, beta=beta), 3) for p in [1, 2, 3, 4, 5]]
    print(f'beta={beta}:', values)


## 7. Superlinear Production Value

Yeni eklenen configurable production scaling:

```text
production_value = production * (production / reference) ** beta
```

Default `beta=0.0` olduğu için eski davranış birebir korunur.

Ama `beta > 0` verilirse high-production planetler lineerden daha değerli hale gelir. Örneğin `reference=3.0`, `beta=0.5` için production `3` aynı kalır; production `5` belirgin artar, production `1` düşer.

Dikkat: deneylerde `beta=0.5` mevcut checkpoint üzerine direkt uygulanınca PPO instability ve no-op collapse görüldü. Bu yüzden güvenli başlangıç için `0.1-0.2`, düşük learning rate, yüksek entropy ve küçük clip range tercih edilmeli.


## 8. Neutral Gift Penalty

Neutral gift penalty, agent'ın neutral planet'i yumuşatıp düşmanın onu almasına sebep olduğu durumları cezalandırır.

Kural:

- Launch target owner neutral olmalı.
- Agent fleet'i target'a varmış olmalı.
- Planet sonra enemy tarafından capture edilmiş olmalı.
- Penalty damage fraction ile orantılı olmalı.

Formül:

```text
damage_fraction = damage_done / launch_target_ships
penalty = -production_value * remaining_steps * damage_fraction * reward_scale
```

Bu penalty, neutral planeti capture etmeden sadece enemy'ye hazırlamayı caydırır. Aynı `production_reward_value` helper'ını kullandığı için high-production neutral'ı düşmana hediye etmek daha ağır cezalandırılır.


## 9. Fleet Combat Reward

`FleetTracker`, launch edilen fleet'in env'de ne zaman kaybolduğunu/target'a ulaştığını izler ve outcome üretir. `production_simple` için combat reward sadeleştirilmiştir.

Özet davranış:

| Outcome | Reward |
|---|---|
| Self reinforce | `0` |
| Enemy ship killed veya enemy planet capture | `kill_weight * enemy_ships_killed * strength_factor * reward_scale` |
| Destroyed enroute | `-own_waste / my_total_ships_at_launch * reward_scale` |
| Failed neutral attack | production capture reward alamaz; own waste fleet tracker pathinde temsil edilir |

Capture'ın ana ekonomik rewardu burada değil, `ProductionRewardTracker` pathinde verilir. Böylece neutral capture ve enemy capture doğrudan production payback ile ölçülür.


## 10. Defense Deficit ve Defense Hold

Reward sistemi iki farklı savunma sinyali içerir.

### Defense deficit penalty

`production_simple` içinde launch anında source planet enemy incoming tehdidi altındaysa ve launch sonrası source yeterli savunmayı kaybediyorsa penalty uygulanır. Bu, agent'ın saldırırken kendi planetini açık bırakmasını cezalandırır.

### Defense hold reward

No-op actionları da tracker'a kaydedilebilir. Eğer agent bekleyerek incoming enemy fleet'i absorbe ederse, hold transition'a delayed defense reward yazılır. Bu sistem özellikle `current/aggressive` reward pathlerinde daha aktiftir.

Ama dikkat: no-op'u fazla rewardlamak policy'yi pasifliğe itebilir. Bu yüzden defense hold ağırlığı dikkatli kullanılmalı.


## 11. Delayed Reward ve Buffer Attribution

Rollout sırasında her per-source decision ayrı bir `Transition` olarak `RolloutBuffer` içine eklenir. Transition şunları taşır:

- observation tensors
- source slot
- target slot
- ship bin
- is_noop
- old log probability
- value estimate
- reward
- done
- turn

Delayed event geldiğinde tracker `(buffer_idx, outcome)` döndürür. Sonra:

```text
buffer.add_delayed_reward(buffer_idx, reward, event_turn=t, discount=delayed_reward_discount)
```

Eğer discount `1.0` ise delay cezası yoktur. Eğer örneğin `0.99` ise launch ile event arasındaki her turn rewardu azaltır.


## 12. PPO Açısından Rewardların Kullanımı

Episode sonunda buffer'daki rewardlar GAE ile advantage/return'a dönüştürülür. PPO update içinde:

```text
ratio = exp(new_log_prob - old_log_prob)
policy_loss = -mean(min(ratio * advantage, clipped_ratio * advantage))
value_loss = MSE / clipped value loss
entropy_bonus = -entropy_coef * entropy
```

Önemli nokta: scripted action curriculum kullanıldığında action env'de gerçekten uygulanır ve `old_log_prob`, o scripted action'ın mevcut policy altında evaluate edilmiş log probability'sidir. Bu sayede PPO, scripted davranışın gerçek outcome rewardunu öğrenebilir.


## 13. Training Loglarını Yorumlama

Training sırasında en kritik metrikler:

| Metrik | Ne anlatır | Tehlike işareti |
|---|---|---|
| `launch/p` | Oyuncu başına ortalama launch | `0` veya çok düşükse no-op collapse |
| `H` | Entropy | `0.0` civarı policy collapse |
| `KL` | PPO policy shift | `>0.1` dikkat, `>1` ciddi instability |
| `clip` | Clipped ratio oranı | Çok yüksekse update fazla agresif |
| `D` | Draw sayısı | Sürekli yüksekse oyun bitiremiyor |
| `valL` | Value loss | Reward scale değişince sıçrayabilir |

Örneğin `production_reward_beta=0.5` doğrudan eski checkpoint üzerine verildiğinde `KL` çok yükselip entropy çökmüş ve `launch/p=0` olmuştu. Bu, reward shape değişikliklerinde küçük beta ve daha konservatif PPO hyperparametrelerinin gerekli olduğunu gösterir.


## 14. Güvenli Deney Reçeteleri

Production beta gibi reward shape değişikliklerinde önerilen güvenli başlangıç:

```bash
orbit_wars_venv/bin/python -m rl.train \
  --resume-from checkpoints_production_simple_payback_kill_d256_l6/latest.pt \
  --iterations 140 \
  --episodes-per-iter 5 \
  --random-num-agents \
  --num-agent-choices 2,4 \
  --device cuda \
  --reward-system production_simple \
  --production-reward-beta 0.2 \
  --production-reward-reference 3.0 \
  --scripted-expansion-turns 60 \
  --scripted-expansion-nearest-k 5 \
  --lr 1e-4 \
  --entropy-coef 0.05 \
  --clip-range 0.1 \
  --checkpoint-dir checkpoints_production_simple_exp_beta02_scripted_safe \
  --log-path checkpoints_production_simple_exp_beta02_scripted_safe/train_log.jsonl \
  --checkpoint-every 5
```

Durdurma kriterleri:

```text
H < 0.1 ve launch/p < 5 birkaç epoch üst üste sürerse durdur.
KL > 0.2 birkaç epoch üst üste sürerse learning rate / clip azalt.
D=5 sürekli ise deterministic replay ve launch timing incelenmeli.
```
